In [1]:
import pandas as pd

df = pd.read_csv(
    "data/rows.csv",
    usecols=["Product", "Consumer complaint narrative"]
)
df = df.dropna()
df.columns = ["label", "text"]
print(len(df))
df.head()

C:\Users\YB Store\AppData\Local\Temp\ipykernel_1800\2348563907.py:3: DtypeWarning: Columns (5: Consumer complaint narrative) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


383564


,label,text
29904,"Credit reporting, credit repair services, or o...",The Summer of XX/XX/2018 I was denied a mortga...
30629,"Credit reporting, credit repair services, or o...",There are many mistakes appear in my report wi...
30735,"Credit reporting, credit repair services, or o...",There are many mistakes appear in my report wi...
30795,"Credit reporting, credit repair services, or o...",There are many mistakes appear in my report wi...
30807,"Credit reporting, credit repair services, or o...",There are many mistakes appear in my report wi...


In [2]:
df["label"].value_counts()

label
Credit reporting, credit repair services, or other personal consumer reports    92378
Debt collection                                                                 86710
Mortgage                                                                        52987
Credit reporting                                                                31588
Student loan                                                                    21810
Credit card or prepaid card                                                     21379
Credit card                                                                     18838
Bank account or service                                                         14885
Checking or savings account                                                     12881
Consumer Loan                                                                    9474
Vehicle loan or lease                                                            5745
Money transfer, virtual currency, or money servi

In [3]:
df["label"] = df["label"].replace({
    "Credit reporting": "Credit reporting, credit repair services, or other personal consumer reports",
    "Credit card": "Credit card or prepaid card",
    "Prepaid card": "Credit card or prepaid card",
    "Bank account or service": "Checking or savings account",
    "Payday loan": "Payday loan, title loan, or personal loan",
    "Money transfers": "Money transfer, virtual currency, or money service",
    "Virtual currency": "Money transfer, virtual currency, or money service",
    "Consumer Loan": "Vehicle loan or lease",
})
df["label"].value_counts()

label
Credit reporting, credit repair services, or other personal consumer reports    123966
Debt collection                                                                  86710
Mortgage                                                                         52987
Credit card or prepaid card                                                      41667
Checking or savings account                                                      27766
Student loan                                                                     21810
Vehicle loan or lease                                                            15219
Money transfer, virtual currency, or money service                                6979
Payday loan, title loan, or personal loan                                         6168
Other financial service                                                            292
Name: count, dtype: int64

In [4]:
import pandas as pd
print(pd.read_csv("data/rows.csv", nrows=3).columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [5]:
# 1. Import libraries
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 2. Load the dataset
df = pd.read_csv("data/rows.csv",
                 usecols=["Date received", "Date sent to company", "Consumer complaint narrative"],
                 low_memory=False)
df.columns = ["received", "sent", "text"]
df = df.dropna()

# Clean the text (same function used in preprocessing)
def clean_text(t):
    t = t.lower()
    t = re.sub(r"\bx{2,}\b", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text"] = df["text"].apply(clean_text)
df = df[df["text"].str.split().str.len() >= 5]

# Create the target: days between receiving the complaint and sending it to the company
df["received"] = pd.to_datetime(df["received"], errors="coerce")
df["sent"] = pd.to_datetime(df["sent"], errors="coerce")
df["days"] = (df["sent"] - df["received"]).dt.days
df = df.dropna(subset=["days"])
df = df[df["days"] >= 0]

# Display first few rows
print(df.head())

# 3. Check dataset info
print(df.info())
print(df["days"].describe())
print(df["days"].value_counts().head(10))

# 4. Remove extreme outliers and define features (X) and target (y)
df = df[df["days"] <= df["days"].quantile(0.99)]
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df["text"])
y = df["days"]

# 5. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Create and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 7. Make predictions
y_pred = model.predict(X_test)

# 8. Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse:.3f}")
print(f"R2 Score: {r2:.3f}")

# 9. Get intercept and coefficients (top 10 words)
print("Intercept:", model.intercept_)
coefficients = pd.DataFrame(model.coef_, vectorizer.get_feature_names_out(), columns=["Coefficient"])
print(coefficients.sort_values("Coefficient", ascending=False).head(10))

# 10. Visualization: distribution of days
plt.hist(df["days"], bins=30, alpha=0.6)
plt.xlabel("Days until sent to company")
plt.ylabel("Number of complaints")
plt.title("Distribution of Days to Reach the Company")
plt.show()

# 11. Optional: Plot Predicted vs Actual values (sample of 3000 points)
idx = np.random.RandomState(42).choice(len(y_test), 3000, replace=False)
plt.scatter(y_test.values[idx], y_pred[idx], alpha=0.3, color="purple")
plt.xlabel("Actual Days")
plt.ylabel("Predicted Days")
plt.title("Actual vs Predicted Days")
plt.show()

ImportError: DLL load failed while importing _arpacklib: An Application Control policy has blocked this file.

In [6]:
import pandas as pd

df = pd.read_csv("data/rows.csv",
                 usecols=["Date received", "Date sent to company"])
df["days"] = (pd.to_datetime(df["Date sent to company"])
              - pd.to_datetime(df["Date received"])).dt.days

print(df["days"].describe())
print(df["days"].value_counts().head(10))

count    1.282355e+06
mean     3.174895e+00
std      1.493060e+01
min     -1.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      1.962000e+03
Name: days, dtype: float64
days
 0    796782
 1    112763
 2     66404
 3     53639
 4     49880
 5     44731
 6     34523
 7     22441
 8      9922
-1      7052
Name: count, dtype: int64


In [7]:
import pandas as pd

df = pd.read_csv("data/rows.csv",
                 usecols=["Date received", "Date sent to company", "Submitted via"])
df["days"] = (pd.to_datetime(df["Date sent to company"])
              - pd.to_datetime(df["Date received"])).dt.days
df = df[(df["days"] >= 0) & (df["days"] <= 30)]

print(df.groupby("Submitted via")["days"].mean().round(2))

Submitted via
Email          3.75
Fax            2.89
Phone          2.54
Postal mail    4.48
Referral       3.91
Web            1.11
Name: days, dtype: float64
